In [ ]:
# ── Colab / Local Setup ─────────────────────────────────────────────────────────
# Run this cell first. In Colab it installs packages and clones the repo.
# Locally it's a no-op if the repo is already on your PYTHONPATH.
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Google Colab detected — installing dependencies (≈1 min first run)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "qdrant-client>=1.7.0",
        "langchain>=1.2.10", "langchain-core>=1.2.15",
        "langchain-openai>=1.1.10", "langchain-community>=0.4.1",
        "langchain-text-splitters>=1.1.1",
        "openai>=1.50.0", "numpy>=1.26.0",
        "sentence-transformers>=2.2.0",
    ], check=True, capture_output=True)
    print("✅ Packages installed")

    repo_path = "/content/VectorSmuggle"
    if not os.path.exists(repo_path):
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/jaschadub/VectorSmuggle.git", repo_path],
            check=True,
        )
        print("✅ Repository cloned")

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    os.chdir(repo_path)
    print("✅ Colab setup complete")
else:
    print("💻 Local environment — using existing installation")

# Vector-Payload Dissociation Demo

This notebook demonstrates the **Vector-Payload Dissociation** technique, a sophisticated steganographic method for hiding sensitive data in vector databases.

## What is Vector-Payload Dissociation?

Vector-Payload Dissociation is a technique where:
1. **Sensitive content** is encoded into a vector using steganographic obfuscation
2. **Benign decoy content** is created to serve as the visible payload
3. The **sensitive vector is paired with the benign payload** in the database
4. Database administrators see only innocent content, while the vector contains hidden data

## Prerequisites

- Qdrant running locally at `http://localhost:6333`
- OpenAI API key configured
- VectorSmuggle framework installed

## Workflow Overview

```mermaid
flowchart TD
    A[Sensitive Financial Report] --> B[Create Steganographic Embedding]
    B --> C[Apply Obfuscation Techniques]
    D[Generate Benign Decoy] --> E[Company Potluck Email]
    C --> F[Pair Sensitive Vector with Benign Payload]
    E --> F
    F --> G[Upload to Qdrant]
    G --> H[View in Dashboard - Only Sees Innocent Content]
    G --> I[Recover Hidden Data with Proper Tools]
```

## Step 1: Setup and Imports

Import all necessary modules and establish connections.

In [ ]:
# ── API Key Configuration (optional) ────────────────────────────────────────────
# The demo works without an API key using free sentence-transformers embeddings.
# For OpenAI embeddings: add OPENAI_API_KEY to Colab Secrets (🔑 icon in the sidebar).
import os

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("✅ OpenAI API key loaded from Colab Secrets")
    else:
        print("ℹ️  No OPENAI_API_KEY secret — will use sentence-transformers (free, no key needed)")
except Exception:
    print("ℹ️  API key not configured — will use sentence-transformers (free, no key needed)")

In [ ]:
from datetime import datetime

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

from steganography.decoys import DecoyGenerator

# VectorSmuggle framework imports
from steganography.obfuscation import EmbeddingObfuscator
from utils.embedding_factory import create_embeddings

print("✅ All imports successful")
print(f"📅 Demo started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 2: Connect to Qdrant

Establish connection to your local Qdrant instance.

In [ ]:
# Connect to Qdrant — falls back to in-memory mode when localhost is unavailable (e.g. Colab)
collection_name = "vector_payload_demo"

try:
    _test = QdrantClient(host="localhost", port=6333, timeout=2)
    _test.get_collections()
    qdrant_client = _test
    print("✅ Connected to local Qdrant")
    print("🌐 Qdrant Dashboard: http://localhost:6333/dashboard")
except Exception:
    qdrant_client = QdrantClient(":memory:")
    print("✅ Using Qdrant in-memory mode (no local server needed)")

print(f"🎯 Demo collection: {collection_name}")

## Step 3: Initialize VectorSmuggle Components

Set up the embedding model, obfuscator, and decoy generator.

In [ ]:
# Initialize embedding model — OpenAI/Ollama if available, sentence-transformers otherwise
print("🔧 Initializing embedding model...")
try:
    embedding_model = create_embeddings()
    print("✅ Embedding model initialized (OpenAI/Ollama)")
except RuntimeError:
    print("⚠️  OpenAI/Ollama unavailable — loading sentence-transformers (free, no API key)...")
    from sentence_transformers import SentenceTransformer as _ST

    class _STEmbeddings:
        def __init__(self, model="all-MiniLM-L6-v2"):
            self._m = _ST(model)
        def embed_query(self, text):
            return self._m.encode(text).tolist()
        def embed_documents(self, texts):
            return [v.tolist() for v in self._m.encode(texts)]

    embedding_model = _STEmbeddings()
    print("✅ Embedding model initialized (sentence-transformers/all-MiniLM-L6-v2, 384-dim)")

# Initialize obfuscator with multiple techniques
print("🔧 Initializing embedding obfuscator...")
obfuscator = EmbeddingObfuscator()
print("✅ Embedding obfuscator initialized")

# Initialize decoy generator
print("🔧 Initializing decoy generator...")
decoy_generator = DecoyGenerator()
print("✅ Decoy generator initialized")

print("\n🚀 All components ready for Vector-Payload Dissociation")

## Step 4: Prepare Sensitive Content

Create realistic sensitive financial data that we want to hide.

In [ ]:
# Sensitive financial report content
sensitive_content = """
CONFIDENTIAL FINANCIAL REPORT - Q3 2024
========================================

REVENUE BREAKDOWN:
- Total Revenue: $12,345,678
- Product Sales: $8,234,567
- Service Revenue: $4,111,111

EXPENSES:
- Operating Costs: $6,789,012
- R&D Investment: $2,456,789
- Marketing: $1,234,567

NET PROFIT: $1,865,310

SENSITIVE NOTES:
- Acquisition target: TechCorp Inc. ($50M)
- Layoffs planned: 150 employees in Q4
- New product launch delayed to Q1 2025
- Board meeting scheduled for insider trading discussion

CONFIDENTIAL - DO NOT DISTRIBUTE
""".strip()

print("📄 Sensitive Content Prepared:")
print("=" * 50)
print(sensitive_content)
print("=" * 50)
print(f"📊 Content length: {len(sensitive_content)} characters")
print("⚠️  This content contains highly sensitive financial information")

## Step 5: Create Steganographic Embedding

Convert the sensitive content into an embedding and apply obfuscation techniques.

In [ ]:
# Create embedding from sensitive content
print("🔄 Creating embedding from sensitive content...")
original_embedding = embedding_model.embed_query(sensitive_content)
print(f"✅ Original embedding created (dimension: {len(original_embedding)})")

# Convert to numpy array for obfuscation
original_embedding_array = np.array(original_embedding).reshape(1, -1)

# Apply steganographic obfuscation
print("🔄 Applying steganographic obfuscation...")
obfuscation_result = obfuscator.obfuscate(
    original_embedding_array,
    techniques=['noise', 'rotation', 'scaling', 'offset']
)

# Extract obfuscated embedding and metadata
obfuscated_embedding = obfuscation_result['embeddings'][0]
obfuscation_metadata = obfuscation_result

print(f"✅ Obfuscation applied with techniques: {obfuscation_result.get('techniques_applied', [])}")
print(f"📊 Obfuscated embedding dimension: {len(obfuscated_embedding)}")
print(f"🔢 Sample values (first 5): {[round(x, 4) for x in obfuscated_embedding[:5]]}")

# Show the difference
difference = np.mean(np.abs(np.array(original_embedding) - np.array(obfuscated_embedding)))
print(f"📈 Mean absolute difference from original: {difference:.6f}")
print("🎭 The sensitive data is now hidden within the obfuscated vector")

## Step 6: Generate Benign Decoy Payload

Create innocent-looking content that will be visible in the database.

In [ ]:
# Generate benign decoy content
print("🔄 Generating benign decoy payload...")
decoy_texts = decoy_generator.generate_decoy_text(category="business", length=300, num_documents=1)
decoy_text = decoy_texts[0] if decoy_texts else "Company potluck event scheduled for Friday!"

# Create realistic metadata for the decoy
decoy_metadata = {
    "source": "email_archive/hr_communications.eml",
    "filename": "company_potluck_2024.eml",
    "file_type": ".eml",
    "timestamp": datetime.now().isoformat(),
    "author": "Jennifer Martinez <j.martinez@company.com>",
    "subject": "Company Potluck Event - Friday!",
    "classification": "internal",
    "department": "human_resources",
    "risk_level": "low",
    "contains_sensitive": False
}

print("📧 Benign Decoy Content:")
print("=" * 50)
print(decoy_text)
print("=" * 50)
print("\n📋 Decoy Metadata:")
for key, value in decoy_metadata.items():
    print(f"  {key}: {value}")

print("\n✅ Benign payload created - appears completely innocent!")
print("🎭 This is what security administrators will see in Qdrant")

## Step 7: Perform Vector-Payload Dissociation

This is the core technique: pair the sensitive vector with the benign payload.

In [ ]:
# Configuration: Choose your mode
STEALTH_MODE = True  # Set to False for demo mode with hidden fields

print("🎯 Performing Vector-Payload Dissociation...")
print(f"🔧 Mode: {'STEALTH' if STEALTH_MODE else 'DEMO'} (hidden fields {'disabled' if STEALTH_MODE else 'enabled'})")

# Create base payload
payload = {
    "page_content": decoy_text,  # Visible innocent content
    "metadata": decoy_metadata   # Innocent metadata
}

# Add hidden fields only in demo mode
if not STEALTH_MODE:
    payload.update({
        "_hidden_technique": "vector_payload_dissociation",
        "_original_content_type": "financial_report",
        "_contains_sensitive": True,
        "_demo_mode": True
    })
    print("📝 Added hidden fields for educational demonstration")
else:
    print("🔒 No hidden fields - perfect stealth mode")

# The key dissociation: sensitive vector + benign payload
dissociated_point = PointStruct(
    id=1,
    vector=obfuscated_embedding.tolist(),  # Contains hidden sensitive data
    payload=payload
)

print("✅ Vector-Payload Dissociation complete!")
print("\n🔍 What happened:")
print("  📊 Vector contains: Obfuscated financial report data")
print(f"  📧 Payload shows: {decoy_text[:50]}...")
print("  🎭 Dissociation: Vector ≠ Payload content")
print(f"  🔧 Hidden fields: {'Present (demo)' if not STEALTH_MODE else 'Absent (stealth)'}")
print("\n⚠️  The sensitive financial data is now completely hidden!")

## Step 8: Upload to Qdrant

Store the dissociated data in Qdrant and prepare for dashboard inspection.

In [ ]:
# Create collection if it doesn't exist
try:
    qdrant_client.delete_collection(collection_name)
    print(f"🗑️  Deleted existing collection: {collection_name}")
except:
    pass

# Create new collection
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=len(obfuscated_embedding),
        distance=Distance.COSINE
    )
)
print(f"✅ Created collection: {collection_name}")

# Upload the dissociated point
print("🔄 Uploading dissociated data to Qdrant...")
qdrant_client.upsert(
    collection_name=collection_name,
    points=[dissociated_point]
)

print("✅ Upload complete!")
print("\n🎯 Vector-Payload Dissociation successfully deployed to Qdrant")
print(f"📊 Collection: {collection_name}")
print(f"🆔 Point ID: {dissociated_point.id}")
print(f"📏 Vector dimension: {len(dissociated_point.vector)}")

## Step 9: 🔍 INSPECT IN QDRANT DASHBOARD

**NOW IS THE TIME TO INSPECT THE RESULTS!**

### Instructions:
1. **Open your browser** and go to: [http://localhost:6333/dashboard](http://localhost:6333/dashboard)
2. **Navigate to the collection** named `vector_payload_demo`
3. **Click on the point** with ID `1`
4. **Observe what you see:**
   - The **payload** shows innocent company potluck content
   - The **metadata** indicates low risk, HR communication
   - The **vector** is just a list of numbers (meaningless without tools)
   - **No indication** of sensitive financial data anywhere!

### What Security Administrators See:
- ✅ **Content**: Company potluck email
- ✅ **Source**: HR communications
- ✅ **Risk Level**: Low
- ✅ **Classification**: Internal, non-sensitive

### What's Actually Hidden:
- 🔒 **Real Content**: Confidential financial report
- 🔒 **Sensitive Data**: Revenue, profit, acquisition plans
- 🔒 **Hidden in**: The vector numbers (steganographically obfuscated)

**Take your time to explore the dashboard before continuing to the next cell.**

In [ ]:
# Retrieve and display what's visible in Qdrant
print("🔍 Retrieving data from Qdrant (simulating dashboard view)...")

# Get the point from Qdrant
points = qdrant_client.scroll(
    collection_name=collection_name,
    limit=10,
    with_vectors=True
)[0]

if points:
    point = points[0]
    print("\n📊 WHAT QDRANT DASHBOARD SHOWS:")
    print("=" * 60)
    print(f"Point ID: {point.id}")
    if point.vector is not None:
        print(f"Vector Length: {len(point.vector)} dimensions")
        print(f"Vector Sample: {[round(x, 3) for x in point.vector[:5]]}...")
    else:
        print("Vector: Not retrieved (use with_vectors=True to see)")
    print("\nPayload Content:")
    print(f"  📧 Text: {point.payload['page_content']}")
    print("\nMetadata:")
    for key, value in point.payload['metadata'].items():
        print(f"  📋 {key}: {value}")
    print("=" * 60)

    print("\n🎭 SECURITY ASSESSMENT:")
    print("  ✅ Content Type: Company communication")
    print("  ✅ Risk Level: Low")
    print("  ✅ Contains Sensitive Data: False")
    print("  ✅ Classification: Internal, non-confidential")
    print("\n🔒 The financial data is completely hidden!")
else:
    print("❌ No points found in collection")

## Step 10: Demonstrate Data Recovery

Show how an attacker with the proper tools can recover the hidden sensitive data.

In [ ]:
print("🔓 Demonstrating sensitive data recovery...")
print("(This simulates an attacker with insider knowledge and tools)")

# Retrieve the obfuscated vector from Qdrant
retrieved_point = qdrant_client.retrieve(
    collection_name=collection_name,
    ids=[1],
    with_vectors=True
)[0]

retrieved_vector = retrieved_point.vector
print("✅ Retrieved obfuscated vector from Qdrant")

# Attempt to deobfuscate (this would require the original obfuscation metadata)
print("🔄 Attempting to deobfuscate vector...")
try:
    # In a real scenario, the attacker would need the obfuscation metadata
    # For demo purposes, we'll use the metadata we stored

    # Create the proper data structure for deobfuscation
    # Qdrant normalizes vectors for COSINE distance — restore original scale before reversing transforms
    obfuscated_norm = float(np.linalg.norm(obfuscated_embedding))
    retrieved_vector_array = np.array(retrieved_vector).reshape(1, -1) * obfuscated_norm
    obfuscated_data = {
        'embeddings': retrieved_vector_array,
        'metadata': obfuscation_metadata['metadata'],
        'techniques_applied': obfuscation_metadata['techniques_applied']
    }

    recovered_vector = obfuscator.deobfuscate(obfuscated_data)
    recovered_vector_flat = recovered_vector.flatten()
    print("✅ Vector deobfuscation successful")

    # Calculate similarity to original
    similarity = np.dot(original_embedding, recovered_vector_flat) / (
        np.linalg.norm(original_embedding) * np.linalg.norm(recovered_vector_flat)
    )
    print(f"📊 Similarity to original embedding: {similarity:.4f}")

    if similarity > 0.8:  # High similarity threshold
        print("\n🚨 SENSITIVE DATA RECOVERY SUCCESSFUL!")
        print("\n📄 RECOVERED CONTENT:")
        print("=" * 50)
        print(sensitive_content)
        print("=" * 50)
        print("\n⚠️  An attacker with proper tools could extract this data!")
    else:
        print("⚠️  Recovery partially successful but data may be corrupted")

except Exception as e:
    print(f"❌ Deobfuscation failed: {e}")
    print("🔒 Data remains protected (obfuscation metadata required)")

## Conclusion

This demonstration showed how **Vector-Payload Dissociation** can be used to hide sensitive data in plain sight within vector databases.

### Key Takeaways:

1. **Perfect Hiding**: Sensitive financial data is completely invisible to database administrators
2. **Innocent Appearance**: Only benign company communications are visible in the dashboard
3. **Steganographic Obfuscation**: Multiple techniques hide data within vector embeddings
4. **Recovery Possible**: Attackers with proper tools can extract the hidden information
5. **Security Gap**: Traditional monitoring tools cannot detect this technique

### Defense Strategies:

- **Vector Analysis**: Monitor for unusual vector patterns or statistical anomalies
- **Embedding Validation**: Verify that vectors match their claimed content
- **Access Controls**: Limit who can upload vectors to databases
- **Audit Trails**: Log all vector database operations
- **Content Verification**: Cross-reference vector content with payload content

### Research Applications:

This technique demonstrates important security considerations for:
- **Vector Database Security**: Understanding attack vectors against embedding stores
- **AI/ML Security**: Protecting machine learning pipelines from data poisoning
- **Red Team Exercises**: Testing organizational defenses against novel attack vectors
- **Security Research**: Developing detection mechanisms for steganographic attacks

---

**⚠️ Ethical Use Only**: This demonstration is for educational and security research purposes. Use responsibly and only in authorized environments.